In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("bank_dwh.db")
cursor = conn.cursor()

print("Database connected")


Database connected


In [2]:
cursor.executescript("""
DROP TABLE IF EXISTS fact_transactions;
DROP TABLE IF EXISTS dim_clients;
DROP TABLE IF EXISTS dim_products;
DROP TABLE IF EXISTS dim_time;

CREATE TABLE dim_clients (
    client_id INTEGER PRIMARY KEY,
    signup_date DATE,
    city TEXT,
    segment TEXT
);

CREATE TABLE dim_products (
    product_id INTEGER PRIMARY KEY,
    product_name TEXT,
    category TEXT
);

CREATE TABLE dim_time (
    time_id INTEGER PRIMARY KEY,
    date DATE,
    month INTEGER,
    year INTEGER
);

CREATE TABLE fact_transactions (
    transaction_id INTEGER PRIMARY KEY,
    client_id INTEGER,
    product_id INTEGER,
    time_id INTEGER,
    amount REAL
);
""")

conn.commit()

print("Tables created")


Tables created


In [3]:
cursor.executescript("""
INSERT INTO dim_clients VALUES
(1, '2025-01-01', 'Moscow', 'mass'),
(2, '2025-01-03', 'SPB', 'premium'),
(3, '2025-01-10', 'Kazan', 'mass'),
(4, '2025-02-01', 'Moscow', 'premium');

INSERT INTO dim_products VALUES
(1, 'Credit Card', 'cards'),
(2, 'Loan', 'loans'),
(3, 'Deposit', 'savings');

INSERT INTO dim_time VALUES
(1, '2025-01-10', 1, 2025),
(2, '2025-01-20', 1, 2025),
(3, '2025-02-05', 2, 2025),
(4, '2025-02-18', 2, 2025);

INSERT INTO fact_transactions VALUES
(1, 1, 1, 1, 5000),
(2, 2, 2, 2, 15000),
(3, 1, 3, 3, 2000),
(4, 3, 1, 3, 3000),
(5, 4, 2, 4, 10000);
""")

conn.commit()

print("Data inserted")


Data inserted


In [4]:
query = """
SELECT t.year, t.month,
COUNT(DISTINCT f.client_id) AS MAU
FROM fact_transactions f
JOIN dim_time t ON f.time_id = t.time_id
GROUP BY t.year, t.month
"""

pd.read_sql(query, conn)


,year,month,MAU
0,2025,1,2
1,2025,2,3


In [5]:
query = """
SELECT t.year, t.month,
SUM(f.amount) * 1.0 / COUNT(DISTINCT f.client_id) AS ARPU
FROM fact_transactions f
JOIN dim_time t ON f.time_id = t.time_id
GROUP BY t.year, t.month
"""

pd.read_sql(query, conn)


,year,month,ARPU
0,2025,1,10000.0
1,2025,2,5000.0


In [6]:
query = """
WITH cohort AS (
    SELECT client_id,
           strftime('%Y-%m', signup_date) AS cohort_month
    FROM dim_clients
),
activity AS (
    SELECT f.client_id,
           strftime('%Y-%m', t.date) AS activity_month
    FROM fact_transactions f
    JOIN dim_time t ON f.time_id = t.time_id
)
SELECT cohort_month,
       activity_month,
       COUNT(DISTINCT activity.client_id) AS active_users
FROM cohort
JOIN activity USING (client_id)
GROUP BY cohort_month, activity_month
ORDER BY cohort_month, activity_month;
"""

pd.read_sql(query, conn)


,cohort_month,activity_month,active_users
0,2025-01,2025-01,2
1,2025-01,2025-02,2
2,2025-02,2025-02,1


In [7]:
with open("metrics.sql", "w") as f:
    f.write(query)

print("SQL saved")


SQL saved
